# 00 · 环境与关系模型（先搞清楚“你在跟谁说话”）

> 这一课不教花哨语法，先把地基打牢：**数据库到底是个什么东西、为什么不是一个 CSV、SQL 在操作的是什么**。
> 你已经会用 Pandas 读 CSV 做 EDA，那本质上是“把整张表读进内存的 DataFrame 里再算”。SQL 走的是另一条路：**你不搬数据，只发一句声明式的提问，数据库自己决定怎么去磁盘上把答案捞出来**。理解这层区别，后面所有“为什么这么写、为什么慢”才有着落。

## 本课地图

1. 一个文件 vs 一个数据库：SQLite 的世界观
2. 关系模型的零件：表 / 行 / 列 / 主键 / 外键 / 约束
3. 范式直觉：为什么业务数据要拆成 5 张表（而不是一张大宽表）
4. NULL 与三值逻辑（面试高频坑，单独讲）
5. 五种主流数据库的定位：SQLite / MySQL / PostgreSQL / SQLServer / Oracle
6. 两种跑 SQL 的姿势：VSCode SQLite 扩展（图形）与 Python sqlite3（代码）

## 1. 一个文件 vs 一个数据库

你做 Superstore EDA 时，手里是一个 `Superstore.csv`。它的特点是：

- 就是一个纯文本文件，打开方式随便（Excel / Pandas / 记事本都行）；
- 没有“结构约束”，列里混进什么类型都没人拦你；
- 你每次用都得**整个读进内存**，数据量大了就吃不消。

`SQLite` 不一样：**一个 `.db` 文件就是一个完整的、自给自足的数据库**。它没有独立的服务进程（不像 MySQL 要起一个后台 daemon），别人打开它，本质上是一个库文件被进程“按规定格式读写”。这就是为什么它叫 “serverless（无服务器）”。

> 类比：CSV 像一张摊在桌上的 Excel 纸，谁都能乱画；SQLite 文件像一本上锁的账本，有目录（索引）、有格式（schema）、有规矩（约束），你只能通过“SQL 这门语言”去翻它。

先连上看看我们的 `shop.db`。

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


上面这段是**全模块通用的启动样板**，每课开头都会用它。注意三件事：

1. 路径用**绝对路径** + 原始字符串 `r"..."`，这样无论从哪个目录打开 notebook，都能找到数据库；
2. 我们用 `pandas.read_sql_query` 把 SQL 结果直接变成 DataFrame——**这是你最舒服的显示层**，你已经会 Pandas，不用再学一套打印表格的 API；
3. 包装成 `q(sql)` 函数，之后 `q("SELECT ...")` 一行就能跑。

In [2]:
# 看看 shop.db 里到底有哪些“表”和“索引”
q('''
SELECT type, name
FROM sqlite_master
WHERE type IN ('table','index')
ORDER BY type, name
''')

,type,name
0,index,idx_items_order
1,index,idx_items_product
2,index,idx_orders_customer
3,index,idx_orders_date
4,table,customers
5,table,employees
6,table,order_items
7,table,orders
8,table,products


`sqlite_master` 是 SQLite 的“系统目录”，记录这个数据库里所有表、索引、视图。能看到我们有 5 张业务表 + 4 个索引。

## 2. 关系模型的零件

打开一张表看看真面目。

In [3]:
# 商品表前 5 行长什么样
q('''
SELECT * FROM products
LIMIT 5
''')

,product_id,product_name,category,subcategory,unit_price,unit_cost
0,1,椅子-01号,家具,椅子,340.54,258.91
1,2,椅子-02号,家具,椅子,283.74,159.76
2,3,椅子-03号,家具,椅子,352.86,195.41
3,4,椅子-04号,家具,椅子,434.40,312.27
4,5,桌子-01号,家具,桌子,992.28,736.16


In [4]:
# 用 PRAGMA 看每张表的“结构定义”：列名、类型、是否可空、主键标记
q("PRAGMA table_info(products)")

,cid,name,type,notnull,dflt_value,pk
0,0,product_id,INTEGER,0,None,1
1,1,product_name,TEXT,1,None,0
2,2,category,TEXT,1,None,0
3,3,subcategory,TEXT,1,None,0
4,4,unit_price,REAL,1,None,0
5,5,unit_cost,REAL,1,None,0


对照上面这张结构表，建立几个核心概念：

| 概念 | 在 products 表里对应什么 | 编程类比 |
|---|---|---|
| **表 (table)** | `products` | 一个类 / 一个二维数组 |
| **行 (row / tuple)** | 一件具体商品，如“椅子-01号” | 一个对象实例 / 一条记录 |
| **列 (column / attribute)** | `product_name`、`unit_price` | 类的字段 |
| **主键 (primary key)** | `product_id` | 唯一 ID，绝不重复、不为空 |
| **外键 (foreign key)** | 在 `order_items.product_id` 里指向 `products.product_id` | 指针 / 引用 |

**主键**是“这一行的身份证”。`product_id = 1` 全库唯一，不存在第二件。这就是为什么订单明细里不存商品名字，只存一个 `product_id` 数字——省空间，而且名字改了不用改订单。

**外键**是“跨表的引用”。`order_items.product_id` 必须在 `products.product_id` 里真实存在，不然就是买了一个不存在的商品。这种“引用必须合法”的规矩叫**参照完整性**。

In [5]:
# 外键是真的会被强制的吗？SQLite 默认是“关闭”外键校验的，要手动打开。
# 看建库脚本里那句 PRAGMA foreign_keys=ON。我们这里演示一下：
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")
try:
    # 故意插一条指向不存在商品的订单明细 -> 应该报错
    cur.execute('''INSERT INTO order_items(item_id, order_id, product_id, quantity, sales, profit)
                   VALUES (999999, 1, 9999, 1, 100, 50)''')
    print("居然插进去了？外键没生效")
except sqlite3.IntegrityError as e:
    print("被外键拦下了，符合预期 ->", e)

被外键拦下了，符合预期 -> FOREIGN KEY constraint failed


> 这个坑你迟早会踩：**SQLite 默认不强制外键**（为了兼容老行为）。生产里连 SQLite 都应该 `PRAGMA foreign_keys = ON;`。MySQL / PostgreSQL 默认就是强制的。

**约束 (constraint)** 就是列上写的那些规矩：`NOT NULL`（不许空）、`PRIMARY KEY`（主键）、`REFERENCES`（外键）、`UNIQUE`、默认值。它们是数据库替你做的数据质量防线，比“在应用层记得判空”可靠得多。

## 3. 范式直觉：为什么是 5 张表，而不是一张大宽表

新手最自然的想法是：“下单信息嘛，一张表存下不就完了？” 我们看看如果硬塞成一张宽表会怎样。

In [6]:
# 订单 + 客户 + 商品 挤在一起的“反规范化宽表”长这样
q('''
SELECT o.order_id, o.order_date,
       c.name      AS customer_name, c.region,
       p.product_name, p.category,
       oi.quantity, oi.sales
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
LIMIT 6
''')

,order_id,order_date,customer_name,region,product_name,category,quantity,sales
0,1,2025-06-15,谢涛,华南,电脑-04号,科技产品,6,43202.52
1,1,2025-06-15,谢涛,华南,显示器-02号,科技产品,3,4107.96
2,2,2025-02-10,林勇,西南,床-04号,家具,4,5033.88
3,2,2025-02-10,林勇,西南,标签-02号,办公用品,1,24.05
4,2,2025-02-10,林勇,西南,标签-04号,办公用品,6,132.19
5,3,2025-09-04,朱平,华南,耳机-04号,科技产品,6,1316.45


问题全写在脸上了：

- **冗余**：同一个客户买了 20 单，他的姓名、地区就重复存 20 遍；同一商品被买了几百次，它的单价/成本就重复几百次。
- **更新异常**：客户改了所在城市，你得把这 20 行全改一遍，漏一行就数据打架。
- **插入/删除异常**：一个还没卖出去的新商品，你根本没法把它先录进去（因为没有订单明细行）。

**规范化 (normalization)** 就是用主键/外键把“一类东西”拆到独立表里：客户归 `customers`、商品归 `products`、订单归 `orders`、订单里买了啥归 `order_items`。各自只存一份，靠 ID 互相引用。

> 一句话：**规范化 = 用空间换一致性，用“拆表 + 外键引用”消灭重复**。你做 Pandas 时那点 `.merge()` 的内存开销，在数据库里变成了“连接两张表”——所以后面 03 课讲 JOIN 是重中之重。

什么时候**反规范化**（故意冗余）？报表系统为了快，会预先拼好大宽表。那是性能取舍，不是入门该操心的。

## 4. NULL 与三值逻辑（单独拎出来，因为它是坑王）

编程里你习惯的是二值逻辑：真或假。SQL 里有第三个值：**UNKNOWN**，因为 `NULL` 表示“缺失 / 未知”，它不是一个具体的值，而是“这里空着”。

后果很反直觉：**任何值跟 NULL 比较，结果都不是真也不是假，而是 NULL。**

In [7]:
# 演示三值逻辑：NULL = NULL 结果是什么？
q('''
SELECT
  (1 = 1)        AS one_eq_one,        -- 真
  (1 = 2)        AS one_eq_two,        -- 假
  (NULL = NULL)  AS null_eq_null,      -- 不是 true，是 NULL(未知)
  (NULL = 1)     AS null_eq_one        -- 也是 NULL
''')

,one_eq_one,one_eq_two,null_eq_null,null_eq_one
0,1,0,None,None


In [8]:
# 全库唯一一条 manager_id 为 NULL 的记录：大老板张总没有上级
q("SELECT employee_id, name, title, manager_id FROM employees WHERE manager_id IS NULL")

,employee_id,name,title,manager_id
0,1,张总,总经理,None


In [9]:
# 【经典错误】WHERE manager_id = NULL —— 永远查不出任何行！
# 因为 NULL = NULL 得到的是 NULL(未知)，不被 WHERE 当成“真”
print("用 = NULL 查出来的行数：",
      q("SELECT COUNT(*) c FROM employees WHERE manager_id = NULL")["c"][0])
print("用 IS NULL 查出来的行数：",
      q("SELECT COUNT(*) c FROM employees WHERE manager_id IS NULL")["c"][0])

用 = NULL 查出来的行数： 0
用 IS NULL 查出来的行数： 1


记住两条铁律：

1. **判断空值必须用 `IS NULL` / `IS NOT NULL`，永远不要写 `= NULL`。**
2. `NOT (NULL)` 还是 `NULL`。`WHERE` 只保留“结果为真”的行，凡是算出来是 NULL 的一律被过滤掉——这就是为什么 `= NULL` 查不出东西。

> 你在 C++ 里没有“未知值”这个概念，指针要么有效要么是空指针 `nullptr`。SQL 的 NULL 更像“这里压根没填，连它是不是空都不确定”。这个区别以后写 `WHERE`、`JOIN`、聚合时会反复咬你。

## 5. 五种主流数据库，你以后会遇到谁

| 数据库 | 定位 | 一句话记忆 |
|---|---|---|
| **SQLite** | 嵌入式、单文件、零配置 | 手机、浏览器、桌面软件内置；你本模块全程用它 |
| **MySQL** | 互联网最流行的开源服务库 | 中小型网站默认，生态最大，面试常考 |
| **PostgreSQL** | 功能最强的开源对象-关系库 | 复杂查询、JSON、GIS、窗口函数支持最到位，工程师口碑好 |
| **SQLServer** | 微软企业生态 | .NET / Windows 企业内网 |
| **Oracle** | 重型商业数据库 | 金融、电信等大型企业，要钱、要运维 |

> 学习策略：**SQL 这门语言本身是“长期稳定核心”**，`SELECT/JOIN/窗口函数` 的写法十年后还能用；**具体哪个数据库是“会变的工具”**。所以我们先用 SQLite 把语言学透（零安装、一个 .db 文件就能练），等你要找实习、要搭真实后端时，再按岗位需要切到 MySQL 或 PostgreSQL。语法 90% 通用，方言差异（06 课讲）那 10% 到时候补。

## 6. 两种跑 SQL 的姿势

### 方式 A：Python（本课程默认，可复现、可进 notebook）
就是上面用的：`sqlite3.connect(...)` + `pandas.read_sql_query`。它的好处是**能写进 notebook、能版本管理、能自动化**，也是你以后在数据/AI 项目里真正会用的方式（用 SQL 从库里取数，再喂给 Pandas / 模型）。

### 方式 B：VSCode SQLite 扩展（直观，边写边看）
你已经装了 `alexcvzz.vscode-sqlite`。用法：

1. 在左侧文件栏**右键 `data/shop.db`** → 选 `Open Database`；
2. 侧边栏会出现 `SQLITE EXPLORER`，展开能看到 5 张表，点表名直接看数据；
3. 按 `Ctrl+Shift+P` → 输入 `SQLite: New Query`，开一个 `.sql` 文件，把光标放在任意一句 SQL 里，右键 `Run Query`，结果出现在下方；
4. 关掉：命令面板搜 `SQLite: Close All Databases`。

> 建议：**学语法、随手试的时候用 VSCode 扩展**（所见即所得，不用写 Python 样板）；**做项目、沉淀成作品集时用 notebook / Python**（可复现）。两条腿走路。

下面这段演示“纯 sqlite3 游标”怎么用，让你看清 read_sql_query 背后到底发生了什么。

In [10]:
# 不经过 Pandas，直接用 sqlite3 游标——看清楚底层
cur = conn.cursor()
cur.execute("SELECT name, title, salary FROM employees ORDER BY salary DESC LIMIT 5")
rows = cur.fetchall()          # 拿回一个 list[tuple]
print("高薪 Top5（原始游标结果）：")
for r in rows:
    print("  ", r)

# 参数化查询的正确姿势（防注入，06 课会细讲）：用 ? 占位，不要字符串拼接
cur.execute("SELECT COUNT(*) FROM customers WHERE region = ?", ("华东",))
print("\n华东客户数（参数化）：", cur.fetchone()[0])

高薪 Top5（原始游标结果）：
   ('张总', '总经理', 60000.0)
   ('技术总监', '技术总监', 44259.0)
   ('市场总监', '市场总监', 42742.0)
   ('运营总监', '运营总监', 36161.0)
   ('销售总监', '区域总监', 35615.0)

华东客户数（参数化）： 64


In [11]:
# 收尾：关闭连接（notebook 里其实不强制，但养成习惯）
conn.close()
print("连接已关闭。下一课重新 BOOT 即可。")

连接已关闭。下一课重新 BOOT 即可。


## 本课小结

- SQLite = 一个文件即一个数据库，serverless，零配置，最适合练手。
- 表/行/列是数据；主键是身份证，外键是跨表引用，约束是数据质量防线。
- 规范化把一类东西拆成独立表，消灭冗余——代价是查的时候要 JOIN。
- NULL 是“未知”，导致三值逻辑；判空永远 `IS NULL`，永远别写 `= NULL`。
- SQL 语言本身长期稳定，具体数据库是会变的工具；先 SQLite 学透，再按岗位迁 MySQL/PG。

## 来源与延伸

- SQLite 官方文档（长期稳定核心，权威）：https://www.sqlite.org/docs.html
- SQLite 官方 SELECT 语法：https://www.sqlite.org/lang_select.html
- SQLBolt（交互式入门，浏览器里直接练，强烈推荐配合本模块）：https://sqlbolt.com/
- SQLZoo（另一个老牌交互式练习站）：https://sqlzoo.net/
- Select Star SQL（用真实监狱数据讲“查询背后的逻辑与伦理”，叙事极佳）：https://selectstarsql.com/
- SQLite Tutorial（常被引用的 SQLite 教程站）：https://www.sqlitetutorial.net/